In [155]:
def load_race(race_folder):
    laps_matches      = list(race_folder.glob('*_R_laps.csv'))
    weather_matches   = list(race_folder.glob('*_R_weather.csv'))
    telemetry_matches = list(race_folder.glob('*_telemetry.csv'))

    if not laps_matches:
        return None

    laps = pd.read_csv(laps_matches[0])

    # Merge weather
    if weather_matches:
        weather = pd.read_csv(weather_matches[0])
        laps['LapStartTime_sec'] = pd.to_timedelta(laps['LapStartTime']).dt.total_seconds()
        weather['Time_sec'] = pd.to_timedelta(weather['Time']).dt.total_seconds()
        weather['Rainfall'] = weather['Rainfall'].astype(int)
        laps = pd.merge_asof(
            laps.sort_values('LapStartTime_sec'),
            weather[['Time_sec', 'AirTemp', 'TrackTemp',
                     'Rainfall', 'Humidity', 'WindSpeed']].sort_values('Time_sec'),
            left_on  = 'LapStartTime_sec',
            right_on = 'Time_sec',
            direction = 'nearest'
        )
        laps.drop(columns=['LapStartTime_sec', 'Time_sec'], inplace=True)
    else:
        print(f"No weather file for {race_folder.name} — filling with 0")
        for col in ['AirTemp', 'TrackTemp', 'Rainfall']:
            laps[col] = 0

    # Merge telemetry
    if telemetry_matches:
        tel = pd.read_csv(telemetry_matches[0])
        laps = pd.merge(laps, tel, on=['Driver', 'LapNumber'], how='left')
        print(f"  Merged telemetry for {race_folder.name}")
    else:
        print(f"No telemetry file for {race_folder.name} — filling with 0")
        for col in ['AvgGapAhead', 'MinGapAhead', 'AvgThrottle', 'AvgBrake', 'AvgSpeed']:
            laps[col] = 0

    laps['RaceName'] = race_folder.name
    return laps

In [156]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder

project_root  = Path.cwd().resolve().parent.parent
processed_dir = project_root / 'src' / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

# Load all three seasons
all_race_folders = []
for year in [2023, 2024, 2025]:
    season_dir = project_root / 'src' / 'data' / 'raw' / str(year)
    if season_dir.exists():
        year_folders = sorted(season_dir.iterdir())
        all_race_folders.extend(year_folders)
        print(f"{year}: {len(year_folders)} races found")

print(f"\nTotal races: {len(all_race_folders)}")

# Chronological split, first 2/3 train, last 1/3 test
split_idx = int(len(all_race_folders) * 0.67)
train_races = set(all_race_folders[:split_idx])
test_races = set(all_race_folders[split_idx:])

print(f"Train races: {len(train_races)}")
print(f"Test races:  {len(test_races)}")

train_dfs = []
test_dfs = []

for race_folder in all_race_folders:
    if not race_folder.is_dir():
        continue
    df_race = load_race(race_folder)
    if df_race is None:
        print(f"Skipped {race_folder.name}")
        continue
    if race_folder in train_races:
        train_dfs.append(df_race)
    else:
        test_dfs.append(df_race)
    print(f"Loaded {race_folder.name} — {len(df_race)} laps")

df_train = pd.concat(train_dfs, ignore_index=True)
df_test = pd.concat(test_dfs,  ignore_index=True)

print(f"\nTrain: {len(df_train)} laps across {len(train_dfs)} races")
print(f"Test:  {len(df_test)} laps across {len(test_dfs)} races")

2023: 23 races found
2024: 25 races found
2025: 25 races found

Total races: 73
Train races: 48
Test races:  25
  Merged telemetry for Abu_Dhabi_Grand_Prix
Loaded Abu_Dhabi_Grand_Prix — 1157 laps
  Merged telemetry for Australian_Grand_Prix
Loaded Australian_Grand_Prix — 1003 laps
  Merged telemetry for Austrian_Grand_Prix
Loaded Austrian_Grand_Prix — 1354 laps
  Merged telemetry for Azerbaijan_Grand_Prix
Loaded Azerbaijan_Grand_Prix — 962 laps
  Merged telemetry for Bahrain_Grand_Prix
Loaded Bahrain_Grand_Prix — 1056 laps
  Merged telemetry for Belgian_Grand_Prix
Loaded Belgian_Grand_Prix — 816 laps
  Merged telemetry for British_Grand_Prix
Loaded British_Grand_Prix — 971 laps
  Merged telemetry for Canadian_Grand_Prix
Loaded Canadian_Grand_Prix — 1317 laps
  Merged telemetry for Dutch_Grand_Prix
Loaded Dutch_Grand_Prix — 1343 laps
  Merged telemetry for Hungarian_Grand_Prix
Loaded Hungarian_Grand_Prix — 1252 laps
  Merged telemetry for Italian_Grand_Prix
Loaded Italian_Grand_Prix — 9

In [157]:
def preprocess(df, le_driver=None, le_compound=None, fit=False):
    df = df.copy()

    # Sort chronologically within each driver
    df = df.sort_values(['Driver', 'LapNumber'])

    # Create pit lap flag to identify pit stop rows
    df['PitLap'] = df['PitInTime'].notna().astype(int)

    # Convert timedelta columns
    time_cols = ['LapTime', 'PitOutTime', 'PitInTime',
                 'Sector1Time', 'Sector2Time', 'Sector3Time',
                 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
                 'LapStartTime', 'Time']
    for col in time_cols:
        df[col] = pd.to_timedelta(df[col]).dt.total_seconds()

    # Encode compound before creating target
    # so we can use encoded values for NextCompound target
    if fit:
        le_compound = LabelEncoder()
        all_compounds = list(df['Compound'].astype(str).unique()) + ['Unknown']
        le_compound.fit(all_compounds)

    df['Compound'] = df['Compound'].astype(str).apply(
        lambda x: x if x in le_compound.classes_ else 'Unknown'
    )
    df['Compound'] = le_compound.transform(df['Compound'].astype(str))

    # Create NextCompound target, compound used after the pit stop
    df['NextCompound'] = df.groupby('Driver')['Compound'].shift(-1)

    # Encode driver
    if fit:
        le_driver = LabelEncoder()
        all_drivers = list(df['Driver'].astype(str).unique()) + ['Unknown']
        le_driver.fit(all_drivers)

    df['Driver'] = df['Driver'].astype(str).apply(
        lambda x: x if x in le_driver.classes_ else 'Unknown'
    )
    df['Driver'] = le_driver.transform(df['Driver'].astype(str))

    # Add engineered features from Data to Podium: A Machine Learning Model for Predicting Formula 1 Compound Decisions Max Leischner
    df['LapsSinceLastPit'] = df.groupby('Driver').cumcount()
    df['GapAheadDelta'] = df.groupby('Driver')['AvgGapAhead'].diff().fillna(0)

    # Boolean columns
    df['IsPersonalBest'] = df['IsPersonalBest'].fillna(0).astype(int)

    # Drop leaky and irrelevant columns
    drop_cols = ['LapStartDate', 'TrackStatus', 'Deleted',
             'DeletedReason', 'FastF1Generated', 'DriverNumber',
             'RaceName', 'Team']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    # Create NextCompound target as string names first
    df['NextCompoundName'] = df.groupby('Driver')['Compound'].shift(-1)

    # Filter to pit stop rows only
    pit_df = df[df['PitLap'] == 1].copy()

    # Drop rows where NextCompound is NaN
    pit_df = pit_df.dropna(subset=['NextCompoundName'])

    # Filter to SOFT/MEDIUM/HARD only using encoded values
    # Note: FastF1 encodes tyre compound in all caps
    valid_encoded = [le_compound.transform([c])[0] for c in ['SOFT', 'MEDIUM', 'HARD']
                     if c in le_compound.classes_]
    pit_df = pit_df[pit_df['NextCompoundName'].isin(valid_encoded)]

    # Re-encode consecutively 0, 1, 2
    compound_le = LabelEncoder()
    pit_df['NextCompound'] = compound_le.fit_transform(pit_df['NextCompoundName'])
    pit_df.drop(columns=['NextCompoundName'], inplace=True)

    print(f"Compound classes: {dict(enumerate(compound_le.classes_))}")

    return pit_df, le_driver, le_compound, compound_le

In [158]:
# Running preprocessing steps
df_train, le_driver, le_compound, compound_le = preprocess(df_train, fit=True)
df_test, _, _, _= preprocess(df_test, le_driver=le_driver, le_compound=le_compound, fit=False)

print(f"Train pit stops: {len(df_train)}")
print(f"Test pit stops:  {len(df_test)}")

print(f"\nCompound distribution in test:")
print(df_test['NextCompound'].value_counts())

Compound classes: {0: np.float64(0.0), 1: np.float64(2.0), 2: np.float64(3.0)}
Compound classes: {0: np.float64(0.0), 1: np.float64(2.0), 2: np.float64(3.0)}
Train pit stops: 1697
Test pit stops:  786

Compound distribution in test:
NextCompound
1    356
0    321
2    109
Name: count, dtype: int64


In [159]:
# Save metadata of drivers names and lap numbers
meta_train = df_train[['Driver', 'LapNumber']].copy()
meta_test = df_test[['Driver', 'LapNumber']].copy()

meta_train.to_csv(processed_dir / 'meta_train.csv', index=False)
meta_test.to_csv(processed_dir  / 'meta_test.csv',  index=False)

# Save driver and compound mappings
driver_mapping = {int(code): name for code, name in enumerate(le_driver.classes_)}

compound_mapping = {0: 'HARD', 1: 'MEDIUM', 2: 'SOFT'}

with open(processed_dir / 'compound_mapping.json', 'w') as f:
    json.dump(compound_mapping, f)

with open(processed_dir / 'driver_mapping.json', 'w') as f:
    json.dump(driver_mapping, f)

In [160]:
target = 'NextCompound'
feature_cols = [c for c in df_train.columns if c != target]

X_train = df_train[feature_cols].values.astype(np.float32)
y_train = df_train[target].values.astype(np.int64)      # int64 for CrossEntropyLoss

X_test = df_test[feature_cols].values.astype(np.float32)
y_test = df_test[target].values.astype(np.int64)

In [161]:
from sklearn.model_selection import train_test_split

# Scale features
scaler_X = StandardScaler()
X_train  = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

# Safety net
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

assert np.isnan(X_train).sum() == 0, "NaNs in X_train"
assert np.isnan(X_test).sum()  == 0, "NaNs in X_test"

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    stratify=y_train,   # preserve class balance
    random_state=42
)

In [162]:
# Using SMOTE for class imbalancing
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

In [163]:
import yaml
project_root = Path.cwd().resolve().parent.parent
with open(project_root / 'src' / 'config.yaml') as f:
    config = yaml.safe_load(f)

# Sequences data is only need if networks such as LSTM or GRU are used
def create_sequences(X, y, driver_col_idx, sequence_length=3):
    sequences = []
    labels = []

    unique_drivers = np.unique(X[:, driver_col_idx])

    for driver in unique_drivers:
        mask = X[:, driver_col_idx] == driver
        X_driver = X[mask]
        y_driver = y[mask]

        for i in range(sequence_length, len(X_driver)):
            sequences.append(X_driver[i-sequence_length:i])
            labels.append(y_driver[i])

    return np.array(sequences, dtype=np.float32), np.array(labels, dtype=np.int64)

driver_col_idx = feature_cols.index('Driver')
sequence_length = config['model']['sequence_length']

X_train_seq, y_train_seq = create_sequences(X_train_bal, y_train_bal, driver_col_idx, sequence_length)
X_val_seq, y_val_seq = create_sequences(X_val, y_val, driver_col_idx, sequence_length)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, driver_col_idx, sequence_length)

print(f"X_train_seq: {X_train_seq.shape}")
print(f"X_test_seq:  {X_test_seq.shape}")
print(f"Compounds in train: {len(y_train_seq)}")
print(f"Compounds in test:  {len(y_test_seq)}")

X_train_seq: (1519, 3, 37)
X_test_seq:  (729, 3, 37)
Compounds in train: 1519
Compounds in test:  729


In [164]:
# Save
np.save(processed_dir / 'X_train.npy', X_train_seq)
np.save(processed_dir / 'X_val.npy',   X_val_seq)
np.save(processed_dir / 'X_test.npy',  X_test_seq)
np.save(processed_dir / 'y_train.npy', y_train_seq)
np.save(processed_dir / 'y_val.npy',   y_val_seq)
np.save(processed_dir / 'y_test.npy',  y_test_seq)

print(f"Feature cols: {feature_cols}")
print(f"X_train: {X_train_seq.shape}, y_train: {y_train_seq.shape}")
print(f"X_val:  {X_val.shape},  y_val:  {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"\nClass distribution in train:")

unique, counts = np.unique(y_train_seq, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {compound_mapping[u]}: {c}")


Feature cols: ['Time', 'Driver', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'LapStartTime', 'Position', 'IsAccurate', 'AirTemp', 'TrackTemp', 'Rainfall', 'Humidity', 'WindSpeed', 'AvgGapAhead', 'MinGapAhead', 'AvgThrottle', 'AvgBrake', 'AvgSpeed', 'PitLap', 'LapsSinceLastPit', 'GapAheadDelta']
X_train: (1519, 3, 37), y_train: (1519,)
X_val:  (170, 37),  y_val:  (170,)
X_test:  (786, 37),  y_test:  (786,)

Class distribution in train:
  HARD: 738
  MEDIUM: 584
  SOFT: 197
